# Product Recognition on Store Shelves

**Exam Project:** Developed for the final exam by Massimo Modesti and Federico Tampieri as part of the course.  
This notebook implements a system for recognizing cereal box products on supermarket shelves, following the exam requirements.

---

# Step A - Multiple Product Detection
## 1. Setup and Configuration

In this section, we import necessary libraries, define directories for template and scene images, enumerate template IDs, set thresholds for matching and color filtering, and specify visualization parameters.

In [10]:
import cv2
import numpy as np
import os

# Directories containing template (model) and scene (shelf) images
MODELS_DIR       = "./models/"    # contains files named 0.jpg, 1.jpg, ..., 26.jpg
SCENES_DIR       = './scenes/'    # contains files e1.png to e5.png

MODEL_IDS        = [0, 1, 11, 19, 24, 25, 26]
SCENE_FILES      = ["e1.png", "e2.png", "e3.png", "e4.png", "e5.png"]

# Models to be discriminated with color filter
CONFUSE_MODELS   = {1, 11, 0, 26}
HUE_DIFF_THRESH  = 17  # Allowed Hue difference (degrees)

# Matching and filtering parameters
MIN_MATCHES      = 30     # Minimum number of feature matches/inliers required for a valid detection
RATIO_TEST       = 0.7    # Lowe's ratio test parameter to filter ambiguous matches
MIN_AREA_RATIO   = 0.01   # Minimum bounding-box area ratio relative to scene size
RANSAC_THRESH    = 3.0    # RANSAC reprojection threshold (pixels)

# Visualization styles
BOX_COLOR    = (0, 255, 0)  # green box outlines
CENTER_COLOR = (0, 0, 255)  # red centers
TEXT_COLOR   = (0, 255, 0)  # green text
FONT         = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE   = 0.7
TEXT_THICK   = 2
BOX_THICK    = 10
CIRCLE_RAD   = 4

Now, we add two functions:  
1. `def preprocess_scene(img_scene)`: Converts the image to LAB and applies CLAHE on the L (luminance) channel to boost local contrast without amplifying noise, then converts back to BGR—yielding clearer gradients and textures for more reliable feature detection;
2. `def calc_mean_hue(img_bgr)`: Converts the image to HSV and returns the average Hue (0–180 in OpenCV), providing a compact descriptor of the dominant color used later to disambiguate visually similar packages after geometric matching.

In [11]:
def preprocess_scene(img_scene): 
    lab = cv2.cvtColor(img_scene, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    lab = cv2.merge((cl, a, b))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

def calc_mean_hue(img_bgr): 
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    return float(np.mean(hsv[:, :, 0]))

Initialize SIFT and BFMatcher:

In [12]:
sift = cv2.SIFT_create()
bf   = cv2.BFMatcher(cv2.NORM_L2)

This block loads all reference models and prepares them for matching.
For each `MODEL_ID`, it reads the model image, computes SIFT keypoints and descriptors, records its size, and (for visually similar items listed in `CONFUSE_MODELS`) caches the model’s mean Hue for later color-based disambiguation.
All metadata are stored in the `models` dictionary, which serves as the lookup table used during scene detection.

In [ ]:
# Model loading and preprocessing
models = {} 
for mid in MODEL_IDS: 
    path = os.path.join(MODELS_DIR, f"{mid}.jpg")
    img_model = cv2.imread(path)
    if img_model is None:
        raise FileNotFoundError(f"Model image not found: {path}")

    # keypoints and descriptors
    kp_model, des_model = sift.detectAndCompute(img_model, None)
    h_model, w_model = img_model.shape[:2]

    # mean hue for filtering models
    mean_hue = None
    if mid in CONFUSE_MODELS:
        mean_hue = calc_mean_hue(img_model)

    models[mid] = {
        'img': img_model,
        'kp': kp_model,
        'des': des_model,
        'size': (w_model, h_model),
        'mean_hue': mean_hue
    }

This following block iterates over all shelf **scene images**, detects which **reference products** appear, and reports their location and size.

1. **Load scene & guardrails.**
   Build the file path, read the image, and `raise FileNotFoundError` if it’s missing (fail fast with a clear message).

2. **Preprocess for robust features.**
   Apply `preprocess_scene` (LAB + CLAHE on L) to stabilize local contrast; store scene height/width for later area checks.

3. **Extract scene features.**
   Run `sift.detectAndCompute` to get scene **keypoints** and **descriptors**; print quick diagnostics (count/shapes).

4. **Initialize results.**
   `detections = {}` will collect, per product ID, the list of found instances with geometry and visualization points.

5. **Try each model against the scene.**
   Loop over `models.items()` to retrieve a model’s keypoints, descriptors, size (and optional mean Hue).

6. **Descriptor matching (SIFT + Lowe’s ratio).**
   Use `bf.knnMatch(des_model, des_scene, k=2)`: keep matches where the best distance is < `RATIO_TEST` × second best.
   If **too few good matches** (`< MIN_MATCHES`), skip this model for this scene.

7. **Geometric verification (Homography + RANSAC).**
   Build `src_pts`/`dst_pts` from matched keypoints and estimate the **homography** with RANSAC (`RANSAC_THRESH` reprojection).
   If no homography or **too few inliers** (`< MIN_MATCHES`), skip—this filters out accidental or inconsistent matches.

8. **Project model & derive a rotated box.**
   Project the model’s four corners via the homography; fit a **minimum-area rectangle** to get **center (cx, cy)**, **width/height**, and **angle**.
   Discard detections whose box area is **too small** w.r.t. the scene (`MIN_AREA_RATIO`).

9. **Optional color sanity check (for confusing models).**
   For IDs in `CONFUSE_MODELS`, warp the detected quadrilateral back to the model’s frame, compute the region’s **mean Hue**, and compare it to the model’s mean Hue. If the difference exceeds `HUE_DIFF_THRESH`, skip (likely a false positive).

10. **Record the detection.**
    Append a dict with center, width, height, angle, and box vertices to `detections[mid]`; log the inliers count.

11. **Textual report.**
    After all models are tested, print a short summary per scene: for each product ID, how many instances and their geometry.

12. **Visualization.**
    Draw the rotated bounding boxes, centroids, and product IDs on a copy of the scene, then display the annotated image.


In [ ]:
# Process each scene image
for scene_file in SCENE_FILES:
    scene_path = os.path.join(SCENES_DIR, scene_file)
    img_scene = cv2.imread(scene_path)
    if img_scene is None:
        raise FileNotFoundError(f"Scene image not found: {scene_path}") 

    # Pre-process of the scene
    img_proc = preprocess_scene(img_scene)
    h_scene, w_scene = img_proc.shape[:2]

    # Keypoints e descriptors extraction
    kp_scene, des_scene = sift.detectAndCompute(img_proc, None)
    print(f"\n=== Analisi scena {scene_file} ===")
    print(f"Kp scena: {len(kp_scene)}  Descriptors: {None if des_scene is None else des_scene.shape}")

    detections = {} 
    # Matching and homography for each model 
    for mid, data in models.items(): 
        kp_model = data['kp']
        des_model = data['des']
        w_model, h_model = data['size']

        print(f"\n[Modello {mid}]: kp_modello={len(kp_model)}  des_modello={None if des_model is None else des_model.shape}")
        if des_model is None or des_scene is None:
            print("  => Skip: descrittori assenti")
            continue

        # 1) Matching SIFT + Lowe ratio test 
        matches = bf.knnMatch(des_model, des_scene, k=2) 
        print(f"  Raw matches: {len(matches)}")
        good = [m for m,n in matches if m.distance < RATIO_TEST * n.distance]
        print(f"  Good matches (ratio<{RATIO_TEST}): {len(good)}")
        if len(good) < MIN_MATCHES:
            print(f"  => Skip: too few good matches (<{MIN_MATCHES})")
            continue

        # 2) Homography estimation with RANSAC
        src_pts = np.float32([kp_model[m.queryIdx].pt for m in good]).reshape(-1,1,2)
        dst_pts = np.float32([kp_scene[m.trainIdx].pt for m in good]).reshape(-1,1,2)
        M, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, RANSAC_THRESH)
        if M is None:
            print("  => Skip: omografia non trovata")
            continue
        inliers = int(mask.sum())
        print(f"  Inliers RANSAC: {inliers}")
        if inliers < MIN_MATCHES:
            print(f"  => Skip: pochi inliers (<{MIN_MATCHES})")
            continue

        # 3) Transform model corners
        corners = np.float32([[0,0],[w_model,0],[w_model,h_model],[0,h_model]]).reshape(-1,1,2)
        dst_c  = cv2.perspectiveTransform(corners, M)
        pts    = dst_c.reshape(-1,2).astype(np.float32)

        # 4) Calculate rotated bbox
        rot_rect = cv2.minAreaRect(pts)
        (cx, cy), (w_box, h_box), angle = rot_rect
        box_pts = cv2.boxPoints(rot_rect).astype(int)

        # 5) Minimum area filter
        if w_box * h_box < MIN_AREA_RATIO * (w_scene * h_scene):
            print(f"  => Skip: area troppo piccola (<{MIN_AREA_RATIO*100:.2f}% scena)")
            continue

        # 6) Conditioned color filter
        if mid in CONFUSE_MODELS:
            src_rect = np.float32([[0,0],[w_model,0],[w_model,h_model],[0,h_model]])
            dst_rect = np.float32(box_pts)
            M_inv    = cv2.getPerspectiveTransform(dst_rect, src_rect)
            roi       = cv2.warpPerspective(img_scene, M_inv, (w_model, h_model))
            mean_h_roi = calc_mean_hue(roi)
            diff_h     = abs(mean_h_roi - data['mean_hue'])
            print(f"  Hue ROI={mean_h_roi:.1f}, Modello={data['mean_hue']:.1f}, Δ={diff_h:.1f}")
            if diff_h > HUE_DIFF_THRESH:
                print(f"  => Skip: differenza colore troppo alta (>±{HUE_DIFF_THRESH}°)")
                continue

        # Detection saved
        detections.setdefault(mid, []).append({
            'center': (int(round(cx)), int(round(cy))),
            'width':  int(round(w_box)),
            'height': int(round(h_box)),
            'angle':  angle,
            'box_pts': box_pts
        })
        print(f"  *** Rilevata istanza modello {mid} ({inliers} inliers) ***")

    # Results Report
    print(f"\nRisultati per {scene_file}:")
    if not detections:
        print("  Nessun prodotto riconosciuto.")
    for pid, dets in detections.items():
        print(f"  Modello {pid} - {len(dets)} istanza(e) individuata(e):")
        for idx, det in enumerate(dets, 1):
            cx, cy = det['center']
            w_b, h_b = det['width'], det['height']
            print(f"    Istanza {idx} {{posizione: ({cx},{cy}), w={w_b}px, h={h_b}px, angolo={det['angle']:.1f}°}}")

    # Draw and display results with ID
    vis = img_scene.copy()
    for pid, dets in detections.items():
        for det in dets:
            # rotated bounding box
            cv2.drawContours(vis, [det['box_pts']], 0, BOX_COLOR, BOX_THICK)
            # centroid
            cv2.circle(vis, det['center'], CIRCLE_RAD, CENTER_COLOR, -1)
            # centered ID text
            text = f"ID: {pid}"
            (tw, th), _ = cv2.getTextSize(text, FONT, FONT_SCALE, TEXT_THICK)
            tx = det['center'][0] - tw // 2
            ty = det['center'][1] + th // 2
            cv2.putText(vis, text, (tx, ty), FONT, FONT_SCALE, TEXT_COLOR, TEXT_THICK)

    cv2.imshow(f"Detections - {scene_file}", vis)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


=== Analisi scena e1.png ===
Kp scena: 2630  Descriptors: (2630, 128)

[Modello 0]: kp_modello=8044  des_modello=(8044, 128)
  Raw matches: 8044
  Good matches (ratio<0.7): 322
  Inliers RANSAC: 214
  Hue ROI=43.5, Modello=48.0, Δ=4.5
  *** Rilevata istanza modello 0 (214 inliers) ***

[Modello 1]: kp_modello=3158  des_modello=(3158, 128)
  Raw matches: 3158
  Good matches (ratio<0.7): 142
  Inliers RANSAC: 92
  Hue ROI=26.8, Modello=57.7, Δ=30.9
  => Skip: differenza colore troppo alta (>±17°)

[Modello 11]: kp_modello=1444  des_modello=(1444, 128)
  Raw matches: 1444
  Good matches (ratio<0.7): 126
  Inliers RANSAC: 53
  Hue ROI=27.0, Modello=25.7, Δ=1.2
  *** Rilevata istanza modello 11 (53 inliers) ***

[Modello 19]: kp_modello=1390  des_modello=(1390, 128)
  Raw matches: 1390
  Good matches (ratio<0.7): 3
  => Skip: too few good matches (<30)

[Modello 24]: kp_modello=1083  des_modello=(1083, 128)
  Raw matches: 1083
  Good matches (ratio<0.7): 14
  => Skip: too few good matches 